## Importação das bibliotecas
Nesta etapa, importamos as bibliotecas necessárias para a manipulação, 
análise e visualização dos dados dos pedidos de manutenção.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

**Justificação das bibliotecas utilizadas:**

- **Pandas (pd):** permite carregar, manipular e analisar
  dados tabulares.

- **NumPy (np):** permite realizar operações numéricas
  e matemáticas.

- **Matplotlib (plt):** permite construir gráficos para
  a análise exploratória dos dados.

## 2. Carregamento dos dados de treino

Nesta etapa, carregamos o conjunto de dados de treino
a partir do ficheiro treino.csv.

O objetivo é conhecer a estrutura inicial dos dados
e os atributos que caracterizam os pedidos de manutenção.

Nesta fase, utilizamos exclusivamente os dados de treino,
reservando os dados de validação e de teste para as
etapas posteriores do projeto.

In [ ]:
dados_treino = pd.read_csv("../dados/treino.csv")
dados_treino.head()

In [ ]:
dados_treino.info()

In [ ]:
dados_treino.shape

**Justificação:**

A utilização da função head() permite observar uma
amostra inicial dos dados, identificar os atributos
disponíveis e verificar se o ficheiro foi carregado
corretamente.

A separação dos conjuntos de treino, validação e teste
permite prevenir a fuga de informação (Data Leakage).

## 3. Auditoria dos dados

### 3.1. Identificação dos valores em falta

O objetivo desta análise é identificar os atributos
que apresentam valores em falta, quantificar a sua
ocorrência e avaliar a necessidade do seu tratamento.

In [ ]:
valores_em_falta = dados_treino.isnull().sum()
percentagem_em_falta = (valores_em_falta / len(dados_treino)) * 100

auditoria_faltas = pd.DataFrame({
    "Tipo de dados": dados_treino.dtypes,
    "Valores em falta": valores_em_falta,
    "Percentagem (%)": percentagem_em_falta.round(2)
})

auditoria_faltas

**Justificação e interpretação dos resultados**

A análise dos 1800 pedidos de manutenção permitiu identificar os tipos de dados e quantificar 
os valores em falta em cada atributo do conjunto de treino.

Foram identificados cinco atributos com valores em falta:

- `utilizadores_afetados`: 71 registos (3,94%).
- `alternativa_disponivel`: 73 registos (4,06%).
- `horas_desde_deteccao`: 115 registos (6,39%).
- `reincidencias_30d`: 68 registos (3,78%).
- `idade_anos`: 112 registos (6,22%).

O atributo `horas_desde_deteccao` apresenta a maior percentagem de valores em falta, seguido de `idade_anos`.
Os restantes sete atributos não apresentam valores em falta, incluindo a variável-alvo `urgente` e o identificador `id_pedido`.

Observou-se também que os atributos `utilizadores_afetados` e `reincidencias_30d`, apesar de representarem contagens
inteiras, foram interpretados pelo Pandas como float64 devido à presença de valores em falta.

Nesta fase, decidiu-se preservar os dados originais, sem eliminar registos nem substituir os valores em falta,
uma vez que a ausência de informação não representa necessariamente um valor zero ou uma resposta negativa.

O tratamento destes valores será realizado na fase de preparação dos dados, considerando o tipo e a natureza
de cada atributo, utilizando apenas os dados de treino para ajustar os métodos de imputação durante a comparação
dos modelos, evitando assim a fuga de informação (Data Leakage).

### 3.2. Identificação de duplicados

O objetivo desta etapa é verificar a existência de
identificadores repetidos e registos totalmente duplicados,
de modo a garantir a integridade e a qualidade dos dados
do conjunto de treino.

In [ ]:
ids_duplicados = dados_treino["id_pedido"].duplicated().sum()

print("Número de identificadores duplicados:", ids_duplicados)

**Justificação - Identificadores duplicados**

Foi verificada a existência de identificadores repetidos na coluna `id_pedido`. A análise não identificou duplicados (0 ocorrências em 1 800 registos).

**Decisão:** manter todos os registos. O identificador será utilizado para rastreabilidade e análise de erros, mas não será utilizado como atributo de entrada dos modelos, pois não representa informação relevante para prever a urgência.

In [ ]:
# Contar linhas totalmente duplicadas
linhas_duplicadas = dados_treino.duplicated().sum()

print("Número de linhas totalmente duplicadas:", linhas_duplicadas)

**Observação — Linhas totalmente duplicadas**

A análise dos 1800 registos do conjunto de treino não
identificou linhas totalmente duplicadas, tendo sido
obtidas zero ocorrências.

**Decisão:** manter todos os registos, uma vez que não
foram encontradas duplicações completas que justificassem
a sua eliminação.

Pedidos com identificadores diferentes podem apresentar
características semelhantes sem representarem necessariamente
o mesmo pedido de manutenção.

**Impacto:** nenhum registo foi eliminado, mantendo-se
os 1800 pedidos do conjunto de treino.

### 3.3. Verificação dos domínios dos atributos categóricos

O objetivo desta etapa é identificar as categorias existentes
nos atributos categóricos do conjunto de treino e verificar
se os valores observados correspondem aos domínios esperados.

Esta análise permite identificar categorias inesperadas,
inconsistências e possíveis erros de preenchimento dos dados.

In [ ]:
# Mostrar as categorias existentes em cada atributo

print("Equipamentos:")
print(sorted(dados_treino["equipamento"].dropna().unique()))

print("\nContextos:")
print(sorted(dados_treino["contexto"].dropna().unique()))

print("\nPeríodos:")
print(sorted(dados_treino["periodo"].dropna().unique()))

print("\nInterrupção total:")
print(sorted(dados_treino["interrupcao_total"].dropna().unique()))

print("\nAlternativa disponível:")
print(sorted(dados_treino["alternativa_disponivel"].dropna().unique()))

print("\nSinal de segurança:")
print(sorted(dados_treino["sinal_seguranca"].dropna().unique()))

print("\nUrgência:")
print(sorted(dados_treino["urgente"].dropna().unique()))


**Justificação e interpretação dos resultados**

A análise dos domínios dos atributos categóricos permitiu
confirmar que todos os valores observados pertencem aos
domínios esperados.

Foram identificadas cinco categorias de equipamentos,
quatro categorias de contextos e dois períodos de atividade.

Os atributos interrupcao_total, alternativa_disponivel
e sinal_seguranca apresentam exclusivamente as categorias
"sim" e "nao" entre os valores observados.

A variável-alvo urgente apresenta exclusivamente os valores
0 e 1, correspondentes às classes não urgente e urgente,
respetivamente.

Não foram identificadas categorias inesperadas, erros
de escrita ou valores fora dos domínios esperados.

**Decisão:** preservar todas as categorias originais,
sem realizar correções ou substituições nesta etapa.
Os valores em falta serão tratados posteriormente.

**Justificação:** os valores observados encontram-se
em conformidade com os domínios esperados, não existindo
evidências que justifiquem a sua alteração.

**Impacto:** zero registos alterados ou eliminados,
mantendo-se os 1800 pedidos do conjunto de treino.

### 3.4. Verificação dos domínios e unidades dos atributos numéricos

O objetivo desta etapa é verificar se os valores dos
atributos numéricos respeitam os domínios esperados,
considerando as respetivas unidades de medida.

Pretende-se identificar valores impossíveis ou incoerentes
que possam comprometer a qualidade dos dados e o desempenho
dos modelos de classificação.

In [ ]:
# Definir os atributos numéricos
atributos_numericos = [
    "utilizadores_afetados",
    "horas_desde_deteccao",
    "reincidencias_30d",
    "idade_anos"
]

# Construir uma tabela com os valores mínimos e máximos
auditoria_numerica = pd.DataFrame({
    "Mínimo observado": dados_treino[atributos_numericos].min(),
    "Máximo observado": dados_treino[atributos_numericos].max(),
    "Mínimo esperado": [1, 0.1, 0, 0.1],
    "Máximo esperado": [240, 72, 5, 20],
    "Unidade": ["Pessoas", "Horas", "Ocorrências", "Anos"]
})

# Apresentar os resultados
auditoria_numerica

In [ ]:
# Verificar valores fracionários nos atributos de contagem
atributos_contagem = [
    "utilizadores_afetados",
    "reincidencias_30d"
]

for atributo in atributos_contagem:
    valores = dados_treino[atributo].dropna()
    invalidos = (valores % 1 != 0).sum()

    print(f"{atributo}: {invalidos} valores fracionários")

**Justificação e interpretação dos resultados**

A análise dos atributos numéricos permitiu verificar
que todos os valores observados se encontram dentro
dos limites esperados.

O atributo utilizadores_afetados apresenta valores
entre 1 e 240 pessoas, enquanto reincidencias_30d
apresenta valores entre 0 e 5 ocorrências.

O atributo idade_anos apresenta valores entre 0,1
e 20 anos, respeitando o domínio esperado.

Relativamente ao atributo horas_desde_deteccao,
o valor máximo observado foi de 51,6 horas, inferior
ao limite máximo permitido de 72 horas. Esta diferença
não representa uma inconsistência, uma vez que o valor
observado pertence ao intervalo admissível.

A verificação complementar dos atributos de contagem
não identificou valores fracionários.

**Decisão:** manter os valores numéricos originais,
sem realizar correções ou eliminações nesta etapa.

**Justificação:** não foram identificados valores fora
dos limites esperados nem valores fracionários
nos atributos de contagem.

**Impacto:** zero registos alterados ou eliminados,
preservando-se os 1800 pedidos do conjunto de treino.

### 3.5. Identificação de valores atípicos (Outliers)

O objetivo desta etapa é identificar potenciais valores
atípicos nos atributos numéricos do conjunto de treino,
utilizando o método do intervalo interquartil (IQR).

Pretende-se distinguir valores estatisticamente atípicos
de valores inválidos, evitando a eliminação injustificada
de pedidos de manutenção que possam representar situações
reais e relevantes para a classificação da urgência.

In [ ]:
# Calcular o primeiro e o terceiro quartil
Q1 = dados_treino[atributos_numericos].quantile(0.25)
Q3 = dados_treino[atributos_numericos].quantile(0.75)

# Calcular o intervalo interquartil
IQR = Q3 - Q1

# Calcular os limites estatísticos
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

# Identificar os valores atípicos
outliers = (
    (dados_treino[atributos_numericos] < limite_inferior) |
    (dados_treino[atributos_numericos] > limite_superior)
)

# Organizar os resultados numa tabela
auditoria_outliers = pd.DataFrame({
    "Q1": Q1,
    "Q3": Q3,
    "IQR": IQR,
    "Limite inferior": limite_inferior,
    "Limite superior": limite_superior,
    "Valores atípicos": outliers.sum()
})

# Apresentar os resultados
auditoria_outliers.round(2)

**Justificação e interpretação dos resultados**

A aplicação do método do intervalo interquartil (IQR) permitiu identificar valores atípicos em três dos quatro
atributos numéricos do conjunto de treino.

O atributo utilizadores_afetados apresenta 120 valores atípicos, sendo o atributo com maior número de ocorrências.
Estes valores podem representar pedidos de manutenção que afetam um número elevado de utilizadores.

O atributo horas_desde_deteccao apresenta 59 valores atípicos, correspondentes a pedidos com tempos de
deteção decorridos superiores a 26,1 horas.

O atributo idade_anos apresenta 62 valores atípicos, correspondentes a equipamentos com idade superior
a 13,15 anos.

O atributo reincidencias_30d não apresenta valores atípicos segundo o método utilizado.

No total, foram identificadas 241 ocorrências de valores atípicos, não correspondendo necessariamente a 241
pedidos distintos.

**Decisão:** manter todos os valores atípicos identificados.

**Justificação:** a verificação anterior confirmou que os valores numéricos respeitam os domínios esperados.
Assim, os valores atípicos podem representar situações válidas e relevantes para a classificação da urgência.

A sua eliminação poderia provocar perda de informação e reduzir a capacidade dos modelos para identificar
situações menos frequentes.

**Impacto:** nenhum registo foi alterado ou eliminado, mantendo-se os 1800 pedidos do conjunto de treino.

### 3.6. Consolidação da tabela de auditoria

O objetivo desta etapa é consolidar os resultados obtidos
durante a auditoria do conjunto de treino.

A tabela de auditoria reúne os tipos de dados, as unidades
de medida, os domínios esperados, a quantidade e a percentagem
de valores em falta e os potenciais valores atípicos.

Esta consolidação permite obter uma visão global da qualidade
dos dados e fundamentar as decisões que serão tomadas na
fase de preparação dos dados.

In [ ]:
# Definir os domínios esperados
dominios = {
    "id_pedido": "Identificador único",
    "equipamento": "computador, rede, projetor, climatizacao, eletrico",
    "contexto": "aula, laboratorio, servicos, espaco_comum",
    "utilizadores_afetados": "Inteiros entre 1 e 240",
    "interrupcao_total": "sim, nao",
    "alternativa_disponivel": "sim, nao",
    "horas_desde_deteccao": "0.1 a 72",
    "reincidencias_30d": "Inteiros entre 0 e 5",
    "idade_anos": "0.1 a 20",
    "sinal_seguranca": "sim, nao",
    "periodo": "letivo, nao_letivo",
    "urgente": "0, 1"
}

# Definir as unidades de medida
unidades = {
    "id_pedido": "Não aplicável",
    "equipamento": "Não aplicável",
    "contexto": "Não aplicável",
    "utilizadores_afetados": "Pessoas",
    "interrupcao_total": "Não aplicável",
    "alternativa_disponivel": "Não aplicável",
    "horas_desde_deteccao": "Horas",
    "reincidencias_30d": "Ocorrências",
    "idade_anos": "Anos",
    "sinal_seguranca": "Não aplicável",
    "periodo": "Não aplicável",
    "urgente": "Não aplicável"
}



In [ ]:
# Reutilizar a tabela de valores em falta da secção 3.1
auditoria_final = auditoria_faltas.copy()

# Adicionar as unidades de medida
auditoria_final["Unidade"] = auditoria_final.index.map(unidades)

# Adicionar os domínios esperados
auditoria_final["Domínio esperado"] = auditoria_final.index.map(dominios)

# Adicionar os valores atípicos calculados na secção 3.5
auditoria_final["Valores atípicos"] = (
    auditoria_outliers["Valores atípicos"]
    .reindex(auditoria_final.index)
    .astype("Int64")
)

# Reorganizar as colunas
auditoria_final = auditoria_final[[
    "Tipo de dados",
    "Unidade",
    "Domínio esperado",
    "Valores em falta",
    "Percentagem (%)",
    "Valores atípicos"
]]

pd.set_option("display.max_colwidth", None)
auditoria_final

#### Síntese das decisões de limpeza

| Problema analisado | Evidência | Decisão | Impacto |
|---|---|---|---|
| Valores em falta | 439 valores em falta distribuídos por cinco atributos | Adiar o tratamento para a fase de preparação dos dados | Nenhum registo eliminado |
| Identificadores duplicados | 0 ocorrências | Manter os identificadores originais | Nenhum registo alterado |
| Linhas duplicadas | 0 ocorrências | Manter todos os registos | Nenhum registo eliminado |
| Categorias inesperadas | 0 ocorrências | Preservar as categorias originais | Nenhum registo alterado |
| Valores numéricos inválidos | 0 ocorrências | Manter os valores originais | Nenhum registo alterado |
| Valores fracionários em atributos de contagem | 0 ocorrências | Manter os valores originais | Nenhum registo alterado |
| Valores atípicos | 241 ocorrências em três atributos | Preservar os valores atípicos válidos | Nenhum registo eliminado |

**Justificação e conclusão da auditoria**

A auditoria permitiu avaliar a estrutura, a integridade
e a qualidade dos 1800 pedidos de manutenção presentes
no conjunto de treino.

Foram identificados 439 valores em falta, distribuídos
por cinco atributos, e 241 ocorrências de valores
estatisticamente atípicos.

Estas ocorrências não correspondem necessariamente ao
mesmo número de pedidos distintos, uma vez que um pedido
pode apresentar várias ocorrências.

Os valores atípicos foram preservados por respeitarem
os domínios esperados e poderem representar situações
relevantes para a classificação da urgência.

O tratamento dos valores em falta será realizado na fase
de preparação dos dados, através de métodos de imputação
adequados à natureza de cada atributo.

Durante a auditoria, não foram realizadas alterações
aos dados originais, preservando-se os 1800 registos
do conjunto de treino.

## 4. Análise exploratória e visualização dos dados

### 4.1. Percentagem de valores em falta por atributo

**Objetivo:**

Representar graficamente a percentagem de valores em falta
em cada atributo do conjunto de treino.

**Pergunta de análise:**

Quais são os atributos que apresentam maior percentagem
de valores em falta e que necessitam de tratamento na
fase de preparação dos dados?

In [ ]:
# Criar a figura
plt.figure(figsize=(12, 6))

# Construir o gráfico de barras
plt.bar(
    percentagem_em_falta.index,
    percentagem_em_falta.values,
    color="steelblue",
    edgecolor="black"
)

# Definir o título e os nomes dos eixos
plt.title("Percentagem de valores em falta por atributo")
plt.xlabel("Atributos")
plt.ylabel("Percentagem de valores em falta (%)")

# Definir os limites do eixo vertical
plt.ylim(0, 8)

# Rodar os nomes dos atributos
plt.xticks(rotation=45, ha="right")

# Apresentar a percentagem acima de cada barra
for i, valor in enumerate(percentagem_em_falta.values):
    plt.text(i, valor + 0.1, f"{valor:.2f}%", ha="center")

# Adicionar linhas de referência
plt.grid(axis="y", linestyle="--", alpha=0.4)

# Ajustar a apresentação do gráfico
plt.tight_layout()

# Apresentar o gráfico
plt.show()

**Justificação da escolha do gráfico**

Foi utilizado um gráfico de barras por permitir comparar a percentagem de valores em falta entre os diferentes
atributos do conjunto de treino.

O eixo horizontal representa os atributos e o eixo vertical representa a percentagem de valores em falta, calculada
relativamente aos 1800 pedidos de manutenção.

O eixo vertical inicia-se em zero para evitar distorções na comparação visual das percentagens.

**Interpretação dos resultados**

O atributo horas_desde_deteccao apresenta a maior percentagem de valores em falta (6,39%), seguido de
idade_anos (6,22%).

Os atributos alternativa_disponivel, utilizadores_afetados
e reincidencias_30d apresentam percentagens de 4,06%,
3,94% e 3,78%, respetivamente.

Os restantes sete atributos não apresentam valores em falta.

Esta análise permite identificar os cinco atributos que
necessitam de tratamento na fase de preparação dos dados.

**Limitações**

O gráfico permite identificar a quantidade relativa de valores em falta, mas não permite determinar as causas
da ausência de informação, nem o seu impacto no desempenho dos modelos de classificação.

Também não permite identificar se os valores em falta ocorrem simultaneamente nos mesmos pedidos.

**Decisão**

Preservar os dados originais e realizar o tratamento dos valores em falta posteriormente, através de métodos
de imputação adequados a cada tipo de atributo.

### 4.2. Distribuição dos pedidos de manutenção por equipamento

**Objetivo:**

Analisar a distribuição dos pedidos de manutenção pelas
diferentes categorias de equipamentos presentes no
conjunto de treino.

**Pergunta de análise:**

Quais são os tipos de equipamentos que apresentam maior
e menor número de pedidos de manutenção?

A análise permite compreender a representatividade de
cada categoria no conjunto de dados.

In [ ]:
# Calcular a frequência absoluta por equipamento
frequencia_absoluta = dados_treino["equipamento"].value_counts()

# Calcular a frequência relativa em percentagem
frequencia_relativa = (
    frequencia_absoluta / len(dados_treino)
) * 100

# Organizar os resultados numa tabela
distribuicao_equipamentos = pd.DataFrame({
    "Número de pedidos": frequencia_absoluta,
    "Percentagem (%)": frequencia_relativa.round(2)
})

# Apresentar a tabela
distribuicao_equipamentos

In [ ]:
# Criar a figura
plt.figure(figsize=(10, 6))

# Construir o gráfico de barras
plt.bar(
    frequencia_absoluta.index,
    frequencia_absoluta.values,
    color="steelblue",
    edgecolor="black"
)

# Definir o título e os nomes dos eixos
plt.title("Distribuição dos pedidos de manutenção por equipamento")
plt.xlabel("Tipo de equipamento")
plt.ylabel("Número de pedidos")

# Definir os limites do eixo vertical
plt.ylim(0, frequencia_absoluta.max() * 1.15)

# Apresentar a quantidade e a percentagem acima de cada barra
for i, quantidade in enumerate(frequencia_absoluta.values):

    percentagem = frequencia_relativa.iloc[i]

    plt.text(
        i,
        quantidade + 5,
        f"{quantidade} ({percentagem:.2f}%)",
        ha="center"
    )

# Adicionar linhas de referência
plt.grid(axis="y", linestyle="--", alpha=0.4)

# Ajustar a apresentação
plt.tight_layout()

# Apresentar o gráfico
plt.show()


**Justificação da escolha do gráfico**

Foi utilizado um gráfico de barras por permitir comparar as frequências absolutas entre as diferentes categorias
de equipamentos.

O eixo horizontal representa os tipos de equipamentos, enquanto o eixo vertical representa o número de pedidos
de manutenção.

As etiquetas acima das barras apresentam a frequência absoluta e a frequência relativa de cada categoria.

As percentagens foram calculadas relativamente aos 1800 pedidos do conjunto de treino.

Todos os pedidos foram considerados nesta análise, uma vez que o atributo equipamento não apresenta
valores em falta.

O eixo vertical inicia-se em zero, garantindo uma
comparação visual adequada entre as categorias.

**Limitações**

O gráfico permite analisar a distribuição dos pedidos por equipamento, mas não permite determinar quais
os equipamentos que apresentam maior probabilidade de gerar pedidos urgentes.

Também não permite concluir quais os equipamentos que avariam mais frequentemente, uma vez que não
dispomos de informação sobre a quantidade total de equipamentos existentes no campus.

**Interpretação dos resultados**

A análise da distribuição dos pedidos de manutenção permitiu identificar diferenças na representatividade
das cinco categorias de equipamentos.

A categoria computador apresenta o maior número de pedidos, com 561 ocorrências (31,17%), seguida da
categoria rede, com 415 ocorrências (23,06%).

Em conjunto, estas duas categorias representam 54,22% dos pedidos de manutenção do conjunto de treino.

As categorias climatizacao e projetor apresentam frequências semelhantes, com 328 pedidos (18,22%)
e 327 pedidos (18,17%), respetivamente.

A categoria eletrico apresenta a menor frequência, com 169 pedidos, correspondendo a 9,39% do total.

Esta distribuição demonstra que as categorias não estão representadas de forma uniforme no conjunto
de treino.

Contudo, uma maior frequência de pedidos não significa
necessariamente uma maior probabilidade de urgência.

Esta relação será analisada posteriormente através da comparação da proporção de pedidos urgentes
entre as categorias de equipamentos.

**Decisão**

Manter todas as categorias e todos os registos, preservando a distribuição original do conjunto
de treino.

### 4.3. Distribuição antes e depois da imputação pela mediana

**Objetivo:**

Analisar o impacto da imputação pela mediana na
distribuição do atributo horas_desde_deteccao.

**Pergunta de análise:**

De que forma a substituição dos valores em falta pela
mediana altera a distribuição estatística do atributo?

**Escolha do atributo:**

Foi selecionado o atributo horas_desde_deteccao por
apresentar a maior percentagem de valores em falta
(6,39%) e conter 59 valores estatisticamente atípicos.

A experiência será realizada exclusivamente sobre uma
cópia dos dados de treino, preservando os dados originais.

In [ ]:
# Criar uma cópia do atributo original
horas_original = dados_treino["horas_desde_deteccao"].copy()

# Calcular a mediana dos valores observados
mediana_horas = horas_original.median()

# Substituir os valores em falta pela mediana
horas_imputadas = horas_original.fillna(mediana_horas)

# Apresentar os resultados
print("Mediana:", mediana_horas, "horas")

print("Valores em falta antes:", horas_original.isnull().sum())

print("Valores em falta depois:", horas_imputadas.isnull().sum())

print("Valores observados antes:", horas_original.count())

print("Valores disponíveis depois:", horas_imputadas.count())

In [ ]:
# Comparar as estatísticas antes e depois da imputação
comparacao_imputacao = pd.DataFrame({
    "Antes da imputação": horas_original.describe(),
    "Depois da imputação": horas_imputadas.describe()
})

# Apresentar a tabela
comparacao_imputacao.round(2)

In [ ]:
# Selecionar os valores observados antes da imputação
horas_observadas = horas_original.dropna()

# Definir os mesmos intervalos para os dois histogramas
bins = np.linspace(
    horas_observadas.min(),
    horas_observadas.max(),
    21
)

# Criar uma figura com quatro gráficos
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Histograma antes da imputação
axes[0, 0].hist(
    horas_observadas,
    bins=bins,
    color="steelblue",
    edgecolor="black"
)

axes[0, 0].set_title("Histograma antes da imputação (n=1685)")
axes[0, 0].set_xlabel("Horas desde a deteção")
axes[0, 0].set_ylabel("Número de pedidos")

# Histograma depois da imputação
axes[0, 1].hist(
    horas_imputadas,
    bins=bins,
    color="steelblue",
    edgecolor="black"
)

axes[0, 1].set_title("Histograma depois da imputação (n=1800)")
axes[0, 1].set_xlabel("Horas desde a deteção")
axes[0, 1].set_ylabel("Número de pedidos")

# Garantir a mesma escala nos histogramas
limite_y = max(
    axes[0, 0].get_ylim()[1],
    axes[0, 1].get_ylim()[1]
)

for ax in axes[0]:
    ax.set_ylim(0, limite_y)
    ax.set_xlim(bins[0], bins[-1])

# Boxplot antes da imputação
axes[1, 0].boxplot(horas_observadas)
axes[1, 0].set_title("Boxplot antes da imputação")
axes[1, 0].set_ylabel("Horas desde a deteção")
axes[1, 0].set_xticks([])

# Boxplot depois da imputação
axes[1, 1].boxplot(horas_imputadas)
axes[1, 1].set_title("Boxplot depois da imputação")
axes[1, 1].set_ylabel("Horas desde a deteção")
axes[1, 1].set_xticks([])

# Garantir a mesma escala nos boxplots
for ax in axes[1]:
    ax.set_ylim(0, horas_observadas.max() * 1.05)

# Definir o título geral
fig.suptitle(
    "Distribuição das horas antes e depois da imputação pela mediana",
    fontsize=15
)

# Ajustar e apresentar a figura
plt.tight_layout()
plt.show()

**Justificação da escolha dos gráficos**

Foram utilizados histogramas e boxplots por permitirem analisar diferentes características da distribuição do atributo numérico.

Os histogramas permitem observar a frequência dos valores em diferentes intervalos, enquanto os boxplots permitem
comparar a mediana, os quartis, a dispersão e os potenciais valores atípicos.

Foram utilizados os mesmos intervalos nos histogramas e as mesmas escalas nos gráficos correspondentes,
garantindo uma comparação visual adequada.

Antes da imputação, foram considerados 1685 valores observados e omitidos 115 valores em falta.

Depois da imputação, foram considerados os 1800 valores,
incluindo os 115 valores substituídos pela mediana.

**Interpretação dos resultados**

A mediana calculada para o atributo horas_desde_deteccao foi de 7,1 horas, utilizando exclusivamente os valores
observados no conjunto de treino.

Os histogramas apresentam uma distribuição assimétrica à direita, caracterizada por uma maior concentração
de valores baixos e uma menor frequência de valores elevados.

Após a imputação, observa-se um aumento da concentração de valores no intervalo que contém a mediana.

A média diminuiu de 9,05 para 8,92 horas, enquanto a mediana permaneceu inalterada em 7,1 horas.

O desvio-padrão diminuiu de 7,41 para 7,19 horas, indicando uma redução da dispersão dos dados.

O intervalo interquartil diminuiu de 9 para 8,2 horas, demonstrando uma redução da dispersão na região central
da distribuição.

Os valores mínimo e máximo permaneceram inalterados, correspondendo a 0,1 e 51,6 horas, respetivamente.

O limite superior utilizado na identificação de valores atípicos diminuiu de 26,1 para 24,3 horas.
Assim, alguns valores podem passar a ser considerados estatisticamente atípicos sem terem sido alterados.

**Limitações**

A imputação pela mediana não permite recuperar os verdadeiros valores em falta e pode alterar a distribuição
original, reduzindo artificialmente a variabilidade dos dados e aumentando a concentração de valores
junto da mediana.

Esta experiência não permite concluir, por si só, que a imputação pela mediana melhora o desempenho dos modelos de classificação.

**Decisão**

Considerar a imputação pela mediana como método de tratamento dos valores em falta nos atributos numéricos,
devido à sua menor sensibilidade aos valores extremos.

A imputação realizada nesta etapa tem exclusivamente
uma finalidade exploratória.

Na fase de modelação, a imputação será implementada através de uma Pipeline, respeitando o protocolo de treino, validação e teste.

**Impacto nos dados originais**

Nenhum registo original foi alterado ou eliminado. A experiência foi realizada numa cópia do atributo,
na qual foram imputados 115 valores.

### 4.4. Comparação do número de utilizadores afetados por equipamento

**Objetivo:**

Comparar a distribuição do número de utilizadores afetados
pelos pedidos de manutenção entre as diferentes categorias
de equipamentos.

**Pergunta de análise:**

Existem diferenças no número de utilizadores afetados
pelos pedidos de manutenção entre os diferentes tipos
de equipamentos?

**Escolha dos atributos:**

Foram selecionados os atributos equipamento e
utilizadores_afetados, uma vez que o número de pessoas
afetadas pode constituir informação relevante para
a classificação da urgência dos pedidos.

A análise será realizada exclusivamente sobre os valores
observados no conjunto de treino, sem aplicar imputação
ou eliminar registos dos dados originais.

In [ ]:
# Selecionar os atributos necessários para a análise
dados_comparacao = dados_treino[[
    "equipamento",
    "utilizadores_afetados"
]].copy()

# Agrupar os pedidos por equipamento
grupos_equipamentos = dados_comparacao.groupby("equipamento")

# Calcular o número total de pedidos por equipamento
total_pedidos = grupos_equipamentos.size()

# Calcular o número de valores observados
valores_observados = grupos_equipamentos[
    "utilizadores_afetados"
].count()

# Calcular o número de valores em falta
valores_omitidos = total_pedidos - valores_observados

# Organizar os resultados numa tabela
auditoria_comparacao = pd.DataFrame({
    "Total de pedidos": total_pedidos,
    "Valores observados": valores_observados,
    "Valores omitidos": valores_omitidos
})

# Apresentar a tabela
auditoria_comparacao

In [ ]:
# Calcular as estatísticas por equipamento
estatisticas_equipamentos = grupos_equipamentos[
    "utilizadores_afetados"
].describe()

# Calcular o intervalo interquartil
estatisticas_equipamentos["IQR"] = (
    estatisticas_equipamentos["75%"] -
    estatisticas_equipamentos["25%"]
)

# Apresentar os resultados
estatisticas_equipamentos.round(2)

In [ ]:
# Obter as categorias de equipamentos
categorias = sorted(dados_comparacao["equipamento"].unique())

# Preparar os dados e as etiquetas dos boxplots
dados_boxplot = []
etiquetas = []

for categoria in categorias:

    # Selecionar os valores observados de cada categoria
    valores = dados_comparacao.loc[
        dados_comparacao["equipamento"] == categoria,
        "utilizadores_afetados"
    ].dropna()

    # Adicionar os valores à lista
    dados_boxplot.append(valores)

    # Criar a etiqueta com o número de valores observados
    etiquetas.append(f"{categoria}\n(n={len(valores)})")

# Criar a figura
plt.figure(figsize=(12, 7))

# Construir os boxplots
plt.boxplot(
    dados_boxplot,
    tick_labels=etiquetas,
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.6),
    medianprops=dict(color="darkorange", linewidth=2)
)

# Definir o título e os nomes dos eixos
plt.title("Distribuição dos utilizadores afetados por equipamento")
plt.xlabel("Tipo de equipamento")
plt.ylabel("Número de utilizadores afetados (pessoas)")

# Definir o limite inferior do eixo vertical
plt.ylim(bottom=0)

# Adicionar linhas de referência
plt.grid(axis="y", linestyle="--", alpha=0.4)

# Ajustar e apresentar o gráfico
plt.tight_layout()
plt.show()

**Justificação da escolha do gráfico**

Foi utilizado um gráfico de boxplots agrupados por categoria de equipamento, por permitir comparar
a distribuição de um atributo numérico entre diferentes grupos.

Os boxplots permitem analisar a mediana, os quartis, a dispersão e os potenciais valores atípicos do
número de utilizadores afetados.

O eixo horizontal representa as categorias de equipamentos, enquanto o eixo vertical representa o número de
utilizadores afetados, medido em pessoas.

Foi utilizada a mesma escala vertical para todas as categorias, garantindo uma comparação visual adequada.
As etiquetas apresentam o número de valores observados em cada categoria.

Dos 1800 pedidos do conjunto de treino, foram considerados 1729 valores observados e omitidos 71 valores em falta
(3,94%). Não foi aplicada imputação, uma vez que se pretende analisar a distribuição dos valores efetivamente observados.

**Interpretação dos resultados**

As cinco categorias de equipamentos apresentam distribuições relativamente semelhantes quanto ao
número de utilizadores afetados.

As medianas variam entre 13 e 14 utilizadores, demonstrando uma proximidade entre os valores
centrais das distribuições.

A categoria eletrico apresenta a maior média, com 22,21 utilizadores afetados, enquanto a categoria
computador apresenta a menor média, com 20,33.

A categoria eletrico apresenta também o maior desvio-padrão (29,92), indicando uma maior
dispersão global dos valores.

As categorias climatizacao e projetor apresentam os maiores intervalos interquartis, ambos iguais
a 20 pessoas.

Em todas as categorias, a média é superior à mediana e observam-se valores atípicos superiores, sugerindo
a existência de distribuições assimétricas à direita.

As categorias climatizacao, eletrico e projetor apresentam valores máximos de 240 utilizadores afetados.

A hipótese inicial de que os pedidos relacionados com redes afetariam mais utilizadores do que os
pedidos relacionados com computadores não é sustentada pelos resultados descritivos obtidos.

De forma geral, não se observam diferenças acentuadas entre os valores centrais das distribuições.

**Limitações**

Os boxplots resumem as distribuições através de medidas estatísticas, mas não apresentam todas
as suas características.

As categorias apresentam diferentes números de observações, o que pode influenciar a estabilidade
das estimativas estatísticas.

A omissão dos valores em falta pode influenciar os resultados caso a ausência de informação não
seja aleatória.

Não foram realizados testes de significância estatística, pelo que não é possível concluir
que as distribuições sejam estatisticamente iguais.

Esta análise não permite estabelecer uma relação causal entre o tipo de equipamento, o número de
utilizadores afetados e a urgência dos pedidos.

**Decisão**

Manter os atributos equipamento e utilizadores_afetados para a fase de modelação, uma vez que esta análise
não permite determinar a sua capacidade preditiva relativamente à variável-alvo urgente.

Preservar os valores atípicos, dado que respeitam o domínio esperado e podem representar situações
de manutenção com elevado impacto.

**Impacto nos dados originais**

Nenhum registo original foi alterado ou eliminado.

Os 71 valores em falta foram omitidos exclusivamente
para efeitos de análise estatística e visualização.

### 4.5. Proporção de pedidos urgentes por categoria de equipamento

**Objetivo:**

Analisar a proporção de pedidos urgentes entre as
diferentes categorias de equipamentos do conjunto de treino.

**Pergunta de análise:**

Existem categorias de equipamentos que apresentam
uma maior proporção de pedidos urgentes?

**Escolha dos atributos:**

Foram selecionados os atributos equipamento e urgente,
com o objetivo de explorar a relação entre o tipo
de equipamento e a variável-alvo.

Pretende-se identificar possíveis diferenças na
proporção de urgência entre as categorias, sem
estabelecer relações de causalidade.

A análise será realizada exclusivamente sobre
o conjunto de treino.

In [ ]:
# Agrupar os pedidos por equipamento
grupos_urgencia = dados_treino.groupby("equipamento")

# Calcular o número total de pedidos por equipamento
total_pedidos_urgencia = grupos_urgencia.size()

# Calcular o número de pedidos urgentes
pedidos_urgentes = grupos_urgencia["urgente"].sum()

# Calcular o número de pedidos não urgentes
pedidos_nao_urgentes = (
    total_pedidos_urgencia - pedidos_urgentes
)

# Calcular a percentagem de pedidos urgentes
percentagem_urgentes = (
    pedidos_urgentes / total_pedidos_urgencia
) * 100

# Organizar os resultados numa tabela
auditoria_urgencia = pd.DataFrame({
    "Total de pedidos": total_pedidos_urgencia,
    "Pedidos urgentes": pedidos_urgentes,
    "Pedidos não urgentes": pedidos_nao_urgentes,
    "Percentagem de urgência (%)": percentagem_urgentes
})

# Ordenar pela percentagem de urgência
auditoria_urgencia = auditoria_urgencia.sort_values(
    by="Percentagem de urgência (%)",
    ascending=False
)

# Apresentar os resultados
auditoria_urgencia.round(2)

In [ ]:
# Criar a figura
plt.figure(figsize=(12, 7))

# Construir o gráfico de barras
barras = plt.bar(
    auditoria_urgencia.index,
    auditoria_urgencia["Percentagem de urgência (%)"],
    color="steelblue",
    edgecolor="black"
)

# Adicionar etiquetas acima das barras
for barra, (categoria, linha) in zip(
    barras,
    auditoria_urgencia.iterrows()
):

    percentagem = linha["Percentagem de urgência (%)"]
    total = int(linha["Total de pedidos"])
    urgentes = int(linha["Pedidos urgentes"])

    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 2,
        f"{percentagem:.2f}%\n({urgentes}/{total})",
        ha="center",
        va="bottom"
    )

# Definir o título e os nomes dos eixos
plt.title("Proporção de pedidos urgentes por equipamento")
plt.xlabel("Tipo de equipamento")
plt.ylabel("Percentagem de pedidos urgentes (%)")

# Definir a escala percentual
plt.ylim(0, 100)

# Adicionar linhas de referência
plt.grid(axis="y", linestyle="--", alpha=0.4)

# Ajustar e apresentar o gráfico
plt.tight_layout()
plt.show()

**Justificação da escolha do gráfico**

Foi utilizado um gráfico de barras por permitir comparar
a proporção de pedidos urgentes entre as diferentes
categorias de equipamentos.

O eixo horizontal representa os tipos de equipamentos,
enquanto o eixo vertical representa a percentagem
de pedidos urgentes.

A percentagem foi calculada dividindo o número
de pedidos urgentes pelo número total de pedidos
da respetiva categoria e multiplicando por 100.

As etiquetas acima das barras apresentam a percentagem
de urgência, o número de pedidos urgentes e o número
total de pedidos da categoria.

As categorias foram ordenadas por ordem decrescente
da percentagem de urgência, facilitando a comparação.

O eixo vertical apresenta uma escala entre 0% e 100%,
garantindo uma representação adequada das proporções.

Foram considerados os 1800 pedidos do conjunto de treino,
uma vez que os atributos equipamento e urgente não
apresentam valores em falta.

**Interpretação dos resultados**

A categoria eletrico apresenta a maior proporção
de pedidos urgentes (28,99%), seguida das categorias
rede (27,23%), projetor (25,38%), computador (23,17%)
e climatizacao (20,43%).

A diferença entre a maior e a menor proporção
de urgência é de aproximadamente 8,56 pontos percentuais.

Embora a categoria computador apresente o maior
número absoluto de pedidos urgentes (130), a categoria
eletrico apresenta a maior proporção de urgência,
com 49 pedidos urgentes num total de 169 pedidos.

Este resultado demonstra a importância de distinguir
frequências absolutas de frequências relativas.

As diferenças observadas sugerem que o atributo
equipamento poderá conter informação relevante
para a classificação da urgência.

Contudo, nenhuma categoria apresenta uma proporção
de urgência superior a 50%, pelo que o tipo de
equipamento, isoladamente, não permite determinar
de forma fiável a urgência de um pedido.

Dos 1800 pedidos analisados, 442 são urgentes (24,56%)
e 1358 são não urgentes (75,44%).

Esta distribuição evidencia um desequilíbrio entre
as classes, com predominância da classe não urgente.

Este desequilíbrio deverá ser considerado durante
a avaliação dos classificadores, uma vez que uma
elevada exatidão não garante uma identificação
adequada dos pedidos urgentes.

**Limitações**

A análise é exclusivamente descritiva e não permite
estabelecer relações de causalidade.

As categorias apresentam diferentes números de pedidos,
o que pode influenciar a estabilidade das proporções
calculadas.

Não foram realizados testes de significância estatística
para avaliar as diferenças observadas.

A análise considera apenas a relação entre o tipo
de equipamento e a variável-alvo, não considerando
a influência conjunta dos restantes atributos.

Uma maior proporção de urgência numa categoria
não significa que todos os pedidos dessa categoria
devam ser classificados como urgentes.

**Decisão**

Manter o atributo equipamento como candidato a atributo
de entrada durante a fase de modelação.

A sua capacidade preditiva deverá ser avaliada através
dos classificadores e dos resultados obtidos no
conjunto de validação.

Considerar o desequilíbrio entre as classes durante
a avaliação dos modelos, utilizando as métricas
definidas no enunciado.

**Impacto nos dados originais**

Nenhum registo foi alterado ou eliminado.

Foram utilizados exclusivamente os dados do conjunto
de treino.

## 6. Preparação dos dados

Nesta fase serão preparados os dados para a construção
dos modelos de classificação, nomeadamente Árvore de
Decisão, Random Forest e MLP.

A preparação dos dados respeitará a divisão original
dos conjuntos de treino, validação e teste, evitando
a fuga de informação (data leakage).

### 6.1. Separação dos atributos de entrada e da variável-alvo

**Objetivo:**

Separar os atributos utilizados na previsão da
variável-alvo urgente.

O atributo id_pedido será excluído dos atributos
de entrada, uma vez que representa apenas um
identificador único.

O identificador será preservado para efeitos de
rastreabilidade e posterior análise dos erros.

Inicialmente serão considerados os dez atributos
de entrada disponíveis.

In [ ]:
# Separar os atributos de entrada
X_treino = dados_treino.drop(
    columns=["id_pedido", "urgente"]
)

# Separar a variável-alvo
y_treino = dados_treino["urgente"].copy()

# Preservar os identificadores dos pedidos
ids_treino = dados_treino["id_pedido"].copy()

# Verificar as dimensões
print("Dimensão de X_treino:", X_treino.shape)
print("Dimensão de y_treino:", y_treino.shape)
print("Dimensão de ids_treino:", ids_treino.shape)

In [ ]:
# Apresentar os atributos de entrada
print("Atributos de entrada:")

for atributo in X_treino.columns:
    print(atributo)

# Garantir que os atributos excluídos não estão presentes
assert "id_pedido" not in X_treino.columns
assert "urgente" not in X_treino.columns

print("\nSeparação realizada com sucesso.")

**Justificação**

A separação dos atributos de entrada e da variável-alvo
é necessária para a aprendizagem supervisionada.

Os atributos de entrada representam as características
dos pedidos de manutenção, enquanto a variável urgente
representa a classificação que o modelo deverá aprender
a prever.

A exclusão da variável-alvo dos atributos de entrada
previne a fuga direta de informação.

O identificador id_pedido foi excluído por não representar
uma característica relevante para a previsão da urgência.

**Decisão**

Considerar inicialmente os dez atributos de entrada
disponíveis e avaliar posteriormente as configurações
dos modelos através do conjunto de validação.

**Impacto nos dados originais**

Nenhum registo original foi alterado ou eliminado.
Apenas foram criadas estruturas de dados separadas
para a fase de modelação.

### 6.2. Identificação dos atributos numéricos e categóricos

**Objetivo:**

Identificar e separar os atributos numéricos e
categóricos do conjunto de treino, de forma a
permitir a aplicação de técnicas de preparação
adequadas a cada tipo de atributo.

Esta separação permitirá definir diferentes
estratégias de tratamento dos valores em falta
e de transformação dos dados antes do treino
dos classificadores.

In [ ]:
# Identificar os atributos numéricos
atributos_numericos = X_treino.select_dtypes(
    include=["number"]
).columns.tolist()

# Identificar os atributos categóricos
atributos_categoricos = X_treino.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# Apresentar os atributos numéricos
print("Atributos numéricos:")
for atributo in atributos_numericos:
    print(atributo)

# Apresentar os atributos categóricos
print("\nAtributos categóricos:")
for atributo in atributos_categoricos:
    print(atributo)

# Apresentar o número de atributos
print("\nTotal de atributos numéricos:",
      len(atributos_numericos))

print("Total de atributos categóricos:",
      len(atributos_categoricos))

# Verificar se todos os atributos foram classificados
assert set(atributos_numericos + atributos_categoricos) == set(X_treino.columns)

assert not set(atributos_numericos).intersection(
    atributos_categoricos
)

print("\nIdentificação realizada com sucesso.")

**Justificação**

Os atributos numéricos e categóricos apresentam
características distintas e necessitam de diferentes
técnicas de preparação.

Os atributos numéricos permitem realizar operações
matemáticas e estatísticas, como o cálculo da mediana.

Os atributos categóricos representam categorias
sem uma ordem numérica natural, pelo que não devem
ser convertidos arbitrariamente em valores inteiros
que possam introduzir relações de ordem inexistentes.

**Decisão**

Separar os dez atributos de entrada em dois grupos,
constituídos por quatro atributos numéricos e seis
atributos categóricos.

Esta separação permitirá aplicar estratégias de
imputação e transformação adequadas a cada grupo.

**Impacto nos dados originais**

Nenhum valor foi alterado, eliminado ou substituído.

Foram apenas identificados e agrupados os atributos
de acordo com os seus tipos de dados.

### 6.3. Preparação dos atributos numéricos

**Objetivo:**

Definir uma estratégia de tratamento dos valores em
falta nos atributos numéricos, utilizando a imputação
pela mediana através do SimpleImputer da biblioteca
Scikit-learn.

A estratégia será posteriormente integrada num
Pipeline, garantindo que os parâmetros de imputação
são aprendidos exclusivamente a partir do conjunto
de treino durante a comparação dos modelos.

Para os modelos baseados em árvores, não será
necessário aplicar técnicas de normalização ou
padronização dos atributos numéricos.

In [ ]:
# Importar a ferramenta de imputação
from sklearn.impute import SimpleImputer

# Definir a estratégia de imputação numérica
imputador_numerico = SimpleImputer(
    strategy="median"
)

# Apresentar a configuração
print(imputador_numerico)

**Justificação**

Foi selecionada a imputação pela mediana por ser
uma medida de tendência central menos sensível
a valores extremos do que a média.

A análise exploratória realizada anteriormente
identificou distribuições assimétricas e valores
atípicos válidos nos atributos numéricos, justificando
a utilização desta estratégia.

A imputação permitirá preservar todos os pedidos
de manutenção, evitando a eliminação de registos
devido à existência de valores em falta.

**Decisão**

Utilizar o SimpleImputer com a estratégia median
para o tratamento dos valores em falta nos quatro
atributos numéricos.

Para a Árvore de Decisão e Random Forest, não será
aplicada padronização, uma vez que estes modelos
baseiam as suas decisões em divisões dos atributos
através de limiares.

A aprendizagem das medianas será realizada dentro
do Pipeline, respeitando o protocolo experimental.

**Limitações**

A imputação pela mediana não recupera os verdadeiros
valores em falta e pode reduzir a variabilidade
dos dados.

**Impacto nos dados originais**

Nesta etapa, nenhum valor foi alterado, substituído
ou eliminado. Foi apenas configurada a estratégia
de imputação.

### 6.4. Preparação dos atributos categóricos

**Objetivo:**

Definir as estratégias de tratamento dos valores em
falta e de codificação dos atributos categóricos,
permitindo a sua utilização pelos modelos de
classificação.

Será utilizada a imputação por uma categoria constante
denominada `desconhecido`, seguida da transformação
dos atributos categóricos através do One-Hot Encoding.

Estas operações serão posteriormente integradas
no Pipeline de preparação dos dados.

In [ ]:
# Importar a ferramenta de codificação categórica
from sklearn.preprocessing import OneHotEncoder

# Definir a estratégia de imputação categórica
imputador_categorico = SimpleImputer(
    strategy="constant",
    fill_value="desconhecido"
)

# Definir a estratégia de codificação categórica
codificador_categorico = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Apresentar as configurações
print("Imputador categórico:")
print(imputador_categorico)

print("\nCodificador categórico:")
print(codificador_categorico)

**Justificação**

A análise exploratória identificou 73 valores em falta
no atributo `alternativa_disponivel`, correspondentes
a 4,06% dos pedidos do conjunto de treino.

Foi selecionada a imputação por uma categoria constante,
denominada `desconhecido`, por permitir preservar a
distinção entre a inexistência de uma alternativa e
a ausência de informação sobre a sua disponibilidade.

Foi selecionado o One-Hot Encoding por permitir
representar numericamente os atributos categóricos
sem introduzir relações de ordem inexistentes entre
as categorias.

**Decisão**

Utilizar o SimpleImputer com a estratégia constant,
seguido do OneHotEncoder, para a preparação dos
seis atributos categóricos.

A configuração handle_unknown="ignore" permitirá
transformar categorias não observadas durante o treino.

**Limitações**

A imputação não permite recuperar os verdadeiros
valores em falta.

O One-Hot Encoding aumenta o número de atributos
após a transformação dos dados.

A representação de categorias desconhecidas por zeros
não garante a qualidade das previsões nesses casos.

**Impacto nos dados originais**

Nenhum registo foi alterado ou eliminado nesta etapa.

Foram apenas configuradas as estratégias de imputação
e codificação categórica, que serão posteriormente
integradas no Pipeline.

### 6.5. Construção do ColumnTransformer

**Objetivo:**

Integrar as estratégias de preparação dos atributos
numéricos e categóricos numa única estrutura de
transformação dos dados.

Será utilizado o ColumnTransformer da biblioteca
Scikit-learn, permitindo aplicar diferentes
transformações a diferentes grupos de atributos.

A preparação dos atributos categóricos será
organizada através de um Pipeline, garantindo
que a imputação ocorre antes da codificação.

Esta estrutura será posteriormente integrada
no Pipeline dos classificadores.

In [ ]:
# Importar as ferramentas necessárias
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Construir o Pipeline categórico
pipeline_categorico = Pipeline(
    steps=[
        ("imputacao", imputador_categorico),
        ("codificacao", codificador_categorico)
    ]
)

# Apresentar a configuração
print(pipeline_categorico)

In [ ]:
# Construir o preparador dos dados
preparador_dados = ColumnTransformer(
    transformers=[
        (
            "numericos",
            imputador_numerico,
            atributos_numericos
        ),
        (
            "categoricos",
            pipeline_categorico,
            atributos_categoricos
        )
    ],
    remainder="drop"
)

# Apresentar a configuração
print(preparador_dados)

**Justificação**

Os atributos numéricos e categóricos necessitam
de diferentes estratégias de preparação.

A utilização do ColumnTransformer permite aplicar
transformações específicas a cada grupo de atributos
e combinar os resultados numa única representação
numérica.

O Pipeline categórico garante que o tratamento
dos valores em falta é realizado antes da
codificação das categorias.

Esta abordagem facilita a reutilização das
estratégias de preparação nos diferentes modelos
e contribui para a prevenção da fuga de informação,
quando integrada corretamente no Pipeline dos
classificadores.

**Decisão**

Utilizar o ColumnTransformer para combinar a
imputação numérica pela mediana com a imputação
categórica por uma categoria constante, seguida
do One-Hot Encoding.

As transformações serão ajustadas exclusivamente
ao conjunto de treino durante a comparação dos
modelos.

**Impacto nos dados originais**

Nenhum valor foi alterado, substituído ou eliminado
nesta etapa.

Foi apenas construída a estrutura responsável
pela preparação dos dados.

### 6.6. Preparação do conjunto de validação

**Objetivo:**

Carregar e preparar o conjunto de validação,
preservando a divisão original dos dados
disponibilizada no enunciado.

O conjunto de validação será utilizado para comparar
as diferentes configurações dos classificadores e
os limiares de classificação.

Serão realizadas verificações estruturais para
garantir a compatibilidade entre os conjuntos
de treino e validação, bem como a inexistência
de identificadores comuns.

In [ ]:
# Carregar o conjunto de validação
dados_validacao = pd.read_csv("../dados/validacao.csv")

# Verificar as dimensões
print("Dimensão do conjunto de validação:",
      dados_validacao.shape)

# Verificar o número de identificadores duplicados
print("Identificadores duplicados:",
      dados_validacao["id_pedido"].duplicated().sum())

# Verificar a compatibilidade das colunas
assert dados_validacao.columns.equals(
    dados_treino.columns
)

print("Estrutura das colunas compatível.")

In [ ]:
# Separar os atributos de entrada
X_validacao = dados_validacao.drop(
    columns=["id_pedido", "urgente"]
)

# Separar a variável-alvo
y_validacao = dados_validacao["urgente"].copy()

# Preservar os identificadores dos pedidos
ids_validacao = dados_validacao["id_pedido"].copy()

# Verificar as dimensões
print("Dimensão de X_validacao:", X_validacao.shape)
print("Dimensão de y_validacao:", y_validacao.shape)
print("Dimensão de ids_validacao:", ids_validacao.shape)

In [ ]:
# Identificar pedidos comuns entre treino e validação
ids_comuns = set(ids_treino).intersection(
    ids_validacao
)

# Apresentar o número de identificadores comuns
print("Identificadores comuns:", len(ids_comuns))

# Verificar a integridade dos conjuntos
assert len(dados_validacao) == 600
assert ids_validacao.is_unique
assert ids_validacao.notna().all()
assert len(ids_comuns) == 0

# Verificar os atributos de entrada
assert X_treino.columns.equals(X_validacao.columns)

# Garantir a exclusão do identificador e da variável-alvo
assert "id_pedido" not in X_validacao.columns
assert "urgente" not in X_validacao.columns

print("\nConjunto de validação preparado com sucesso.")

**Justificação**

A utilização de um conjunto de validação independente
permite comparar o desempenho dos classificadores
em dados que não foram utilizados durante a
aprendizagem dos modelos.

As verificações estruturais confirmaram a
compatibilidade dos atributos e a inexistência
de identificadores comuns entre os conjuntos
de treino e validação.

**Decisão**

Preservar a divisão original dos dados fornecida
no enunciado, mantendo os 1800 pedidos de treino
e os 600 pedidos de validação.

O identificador será preservado exclusivamente
para rastreabilidade e análise de erros.

Durante a comparação dos modelos, as transformações
serão ajustadas apenas ao conjunto de treino e
posteriormente aplicadas ao conjunto de validação.

**Impacto nos dados originais**

Nenhum registo foi eliminado ou alterado.

Não foram realizadas operações de imputação,
codificação ou aprendizagem nesta etapa.

O conjunto de teste permanece reservado para
a avaliação final.

### 6.7. Definição das métricas de avaliação

**Objetivo:**

Definir uma função de avaliação comum aos modelos
de referência e aos classificadores desenvolvidos,
garantindo a utilização do mesmo protocolo de
avaliação.

Serão utilizadas as métricas Accuracy, Precision,
Recall, F1-score, taxa de sinalização e custo médio,
bem como os elementos da matriz de confusão.

A classe positiva corresponde aos pedidos urgentes
(urgente = 1).

De acordo com o enunciado, será atribuído um custo
de 4 aos falsos negativos e de 1 aos falsos positivos.

O custo médio será utilizado como principal critério
de seleção da configuração e do limiar de classificação.

In [43]:
# Importar as métricas de avaliação
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Definir os custos dos erros
CUSTO_FN = 4
CUSTO_FP = 1

# Definir a semente aleatória
SEMENTE = 42

print("Custo do falso negativo:", CUSTO_FN)
print("Custo do falso positivo:", CUSTO_FP)
print("Semente aleatória:", SEMENTE)

Custo do falso negativo: 4
Custo do falso positivo: 1
Semente aleatória: 42


In [44]:
# Definir uma função comum de avaliação
def avaliar_modelo(y_real, y_previsto):

    # Converter os dados em arrays
    y_real = np.asarray(y_real)
    y_previsto = np.asarray(y_previsto)

    # Verificar a validade dos dados
    assert len(y_real) == len(y_previsto)
    assert len(y_real) > 0
    assert np.isin(y_real, [0, 1]).all()
    assert np.isin(y_previsto, [0, 1]).all()

    # Construir a matriz de confusão
    matriz = confusion_matrix(
        y_real,
        y_previsto,
        labels=[0, 1]
    )

    # Extrair os elementos da matriz
    tn, fp, fn, tp = matriz.ravel()

    # Calcular as métricas
    accuracy = accuracy_score(
        y_real, y_previsto
    )

    precision = precision_score(
        y_real, y_previsto,
        zero_division=0
    )

    recall = recall_score(
        y_real, y_previsto,
        zero_division=0
    )

    f1 = f1_score(
        y_real, y_previsto,
        zero_division=0
    )

    # Calcular a taxa de sinalização
    taxa_sinalizacao = (tp + fp) / len(y_real)

    # Calcular o custo médio
    custo_medio = (
        CUSTO_FN * fn + CUSTO_FP * fp
    ) / len(y_real)

    # Organizar os resultados
    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "Taxa de sinalização": taxa_sinalizacao,
        "Custo médio": custo_medio,
        "Precision por convenção": (tp + fp == 0),
        "Recall por convenção": (tp + fn == 0),
        "F1 por convenção": (2 * tp + fp + fn == 0)
    }

In [ ]:
# Criar classificações fictícias
y_real_exemplo = [0, 0, 0, 1, 1, 1]
y_previsto_exemplo = [0, 0, 1, 0, 1, 1]

# Avaliar o exemplo
resultado_exemplo = avaliar_modelo(
    y_real_exemplo,
    y_previsto_exemplo
)

# Apresentar os resultados
pd.DataFrame([resultado_exemplo]).T

**Justificação**

Foi desenvolvida uma função de avaliação comum
para garantir que todos os modelos são avaliados
através das mesmas métricas e critérios.

A Accuracy, quando analisada isoladamente, pode
não ser suficiente para avaliar a qualidade da
classificação, especialmente quando existe um
desequilíbrio entre as classes.

A utilização da Precision, Recall e F1-score
permite analisar o desempenho relativamente
à identificação dos pedidos urgentes.

A taxa de sinalização permite avaliar a proporção
de pedidos encaminhados como urgentes.

O custo médio considera as diferentes consequências
dos erros de classificação, penalizando os falsos
negativos quatro vezes mais do que os falsos positivos.

**Decisão**

Utilizar a mesma função de avaliação para todos
os modelos de referência e classificadores.

Adotar o custo médio como principal critério
de seleção, respeitando o protocolo do enunciado.

Utilizar zero_division=0 e identificar explicitamente
as situações em que uma métrica assume zero
por convenção.

**Limitações**

O custo atribuído aos erros constitui uma convenção
definida no enunciado, não representando um custo
monetário real.

**Reprodutibilidade**

Foi definida a semente aleatória 42 para as
experiências que envolvem processos aleatórios.

### 6.8. Construção dos modelos de referência (Baselines)

**Objetivo**

Construir modelos de referência simples que permitam
estabelecer uma base de comparação para os
classificadores de aprendizagem automática.

Serão consideradas três estratégias:

- B01: previsão da classe maioritária.
- B02: classificação de todos os pedidos como urgentes.
- B03: classificação através de uma regra heurística.

Os modelos de referência serão avaliados no conjunto
de validação, utilizando a função de avaliação
definida na secção anterior.

O conjunto de teste permanecerá reservado para
a avaliação final.

#### 6.8.1. Baseline B01 — Classe maioritária

Esta estratégia consiste em identificar a classe
mais frequente no conjunto de treino e atribuir
essa classificação a todos os pedidos do conjunto
de validação.

A classe maioritária será determinada exclusivamente
a partir do conjunto de treino, evitando a utilização
de informação do conjunto de validação na definição
da estratégia.

In [ ]:
# Calcular a frequência absoluta das classes
frequencia_classes = y_treino.value_counts().sort_index()

# Calcular a frequência relativa das classes
percentagem_classes = (
    frequencia_classes / len(y_treino)
) * 100

# Organizar os resultados numa tabela
distribuicao_classes = pd.DataFrame({
    "Número de pedidos": frequencia_classes,
    "Percentagem (%)": percentagem_classes
})

# Apresentar a distribuição das classes
distribuicao_classes.round(2)

In [ ]:
# Identificar a classe maioritária no treino
classe_maioritaria = y_treino.mode().iloc[0]

# Criar as previsões para o conjunto de validação
previsoes_b01 = np.full(
    len(X_validacao),
    classe_maioritaria
)

# Apresentar os resultados
print("Classe maioritária:", classe_maioritaria)

print("Número de previsões:", len(previsoes_b01))

print("Classes previstas:", np.unique(previsoes_b01))

In [ ]:
# Avaliar o baseline da classe maioritária
resultado_b01 = avaliar_modelo(
    y_validacao,
    previsoes_b01
)

# Apresentar os resultados
pd.DataFrame(
    [resultado_b01],
    index=["B01 - Classe maioritária"]
).T

**Justificação**

O baseline da classe maioritária foi utilizado como
estratégia de referência por representar uma abordagem
simples de classificação, sem recorrer às características
dos pedidos de manutenção.

A classe maioritária foi determinada exclusivamente
a partir do conjunto de treino, no qual 75,44% dos
pedidos pertencem à classe não urgente (0).

Esta estratégia permite estabelecer uma referência
para a comparação com os modelos de aprendizagem
automática.

**Interpretação dos resultados**

O baseline classificou os 600 pedidos do conjunto
de validação como não urgentes.

Foram obtidos 453 verdadeiros negativos,
147 falsos negativos, zero falsos positivos
e zero verdadeiros positivos.

A Accuracy foi de 75,5%, enquanto o Recall,
a Precision e o F1-score apresentaram valores
iguais a zero.

A Precision foi definida como zero por convenção,
uma vez que não existem previsões positivas.

A taxa de sinalização foi de 0%, indicando que
nenhum pedido foi classificado como urgente.

O custo médio obtido foi de 0,98, resultante
dos 147 falsos negativos, considerando um custo
de 4 por falso negativo e de 1 por falso positivo.

**Limitações**

Apesar de apresentar uma Accuracy de 75,5%,
esta estratégia não conseguiu identificar nenhum
dos pedidos realmente urgentes.

No contexto da triagem de pedidos de manutenção,
esta limitação poderá provocar atrasos nas
intervenções que deveriam ocorrer no próprio dia.

Os resultados demonstram que a Accuracy,
quando analisada isoladamente, não é suficiente
para avaliar a adequação de um classificador
ao problema em estudo.

**Decisão**

Manter o baseline B01 como referência para a
comparação dos modelos, considerando especialmente
o custo médio, o Recall e a capacidade de
identificação dos pedidos urgentes.